In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

In [ ]:
sys.path.append("../src")
from anomalies import TemporalAnomalyDetector


In [ ]:
# Configuración de Rutas
BASE_DIR = Path.cwd().parent
CSV_CHATS = BASE_DIR / "data" / "processed" / "chats_completos.csv"

# Cargamos el Golden Dataset de la Fase 1
df_chats = pd.read_csv(CSV_CHATS)

In [ ]:
# 2. Pipeline de Detección de Anomalías
# Fijamos un 4% de anomalías esperadas (ajustable según tu criterio forense)
detector = TemporalAnomalyDetector(contamination=0.04) 
df_prepared = detector.prepare_features(df_chats)

# Seleccionamos estrictamente variables de COMPORTAMIENTO (sin texto)
features_to_model = ['Hour', 'Message_Length', 'Time_Delta_Sec']

# Ejecución del Modelo Predictivo
df_anomalies = detector.fit_predict(df_prepared, features_to_model)

In [ ]:
# 3. Métricas Básicas
anomalos = df_anomalies[df_anomalies['Is_Anomaly'] == -1]
normales = df_anomalies[df_anomalies['Is_Anomaly'] == 1]

print(f"\n--- Resumen del Isolation Forest ---")
print(f"Total de eventos evaluados: {len(df_anomalies)}")
print(f"Comportamientos Normales: {len(normales)}")
print(f"Anomalías Detectadas (Crisis/Atípicos): {len(anomalos)}")

In [ ]:
# 4. Visualización Forense (El Gráfico para la Defensa)
plt.figure(figsize=(14, 7))

# Gráfico de dispersión: Timeline vs Hora del Día
sns.scatterplot(
    data=df_anomalies, 
    x='Datetime', 
    y='Hour', 
    hue='Is_Anomaly', 
    palette={1: 'lightgrey', -1: 'red'}, # Rojos para las alertas
    size='Message_Length', # Burbujas grandes = mensajes muy largos (Testamentos)
    sizes=(20, 500),
    alpha=0.8,
    edgecolor='black'
)

plt.title('Detección de Anomalías de Comportamiento (Isolation Forest)', fontsize=15, pad=15)
plt.xlabel('Línea Temporal (Fecha)', fontsize=12)
plt.ylabel('Hora del Día (0-23h)', fontsize=12)

# Ajuste de la leyenda para que no manche los datos
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Estado del Modelo")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Volcado Analítico del Top 10 Anomalías
columnas_mostrar = ['Datetime', 'Emisor', 'Hour', 'Message_Length', 'Time_Delta_Sec', 'Anomaly_Score', 'Mensaje']
top_anomalias = anomalos.sort_values(by='Anomaly_Score').head(10)

print("\n--- TOP 10 EVENTOS MÁS ANÓMALOS MATEMÁTICAMENTE (Mayor a Menor rareza) ---")
display(top_anomalias[columnas_mostrar])